# QK Dot Product Analysis

Pre-softmax attention score structure: statistics, temperature,
positional bias, and content vs position decomposition.

## Why This Matters

QK circuit dot product analyzes the query-key interaction that determines attention patterns. Understanding what features drive attention — positional vs. content-based, local vs. global — reveals how the model decides which information to route where.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.qk_dot_product_analysis import (
    qk_score_statistics, qk_temperature_analysis,
    qk_positional_bias, qk_content_vs_position,
    qk_dot_product_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## QK Score Statistics

Basic statistics of pre-softmax attention scores.

In [ ]:
for layer in range(4):
    result = qk_score_statistics(model, tokens, layer=layer)
    for h in result['per_head']:
        print(f"  L{layer}H{h['head']}: mean={h['mean_score']:.4f}, "
              f"std={h['std_score']:.4f}, range=[{h['min_score']:.3f}, {h['max_score']:.3f}]")

## Temperature Analysis

How sharp are the pre-softmax scores?

In [ ]:
for layer in range(4):
    result = qk_temperature_analysis(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_sharp']} sharp heads")
    for h in result['per_head']:
        flag = ' [SHARP]' if h['is_sharp'] else ''
        print(f"    H{h['head']}: range={h['mean_score_range']:.4f}{flag}")

## Content vs Position

In [ ]:
for layer in range(4):
    result = qk_content_vs_position(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_positional']} positional heads")
    for h in result['per_head']:
        dom = 'POS' if h['is_positional'] else 'CON'
        print(f"    H{h['head']}: pos={h['position_fraction']:.3f}, "
              f"con={h['content_fraction']:.3f} [{dom}]")

## Summary

In [ ]:
result = qk_dot_product_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: sharp={p['n_sharp']}, positional={p['n_positional']}")